In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_npy_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        d = p / "EWS" / "data" / "pretrain" / "npy"
        if d.is_dir():
            return d
    raise FileNotFoundError(
        " EWS/data/pretrain/npy。， NPY_DIR。"
    )


NPY_DIR = find_npy_dir()
print("NPY_DIR:", NPY_DIR)

In [ ]:
N_PER_CLASS = 10
SEED = 42

paths_global = sorted(NPY_DIR.glob("globalwheat_*.npy"))
paths_boni = sorted(NPY_DIR.glob("rgb_bonirob_*.npy"))

rng = random.Random(SEED)


def pick_paths(paths: list[Path], n: int, rng: random.Random) -> list[Path]:
    if len(paths) < n:
        print(f":  {n} ， {len(paths)}  —— 。")
        return list(paths)
    return rng.sample(paths, n)


sel_gw = pick_paths(paths_global, N_PER_CLASS, rng)
sel_rb = pick_paths(paths_boni, N_PER_CLASS, rng)

print(f"globalwheat: {len(paths_global)}  {len(sel_gw)} ")
print(f"rgb_bonirob: {len(paths_boni)}  {len(sel_rb)} ")

In [ ]:
def load_rgb_npy(path: Path) -> np.ndarray:
    x = np.load(path, mmap_mode=None)
    if x.ndim != 3 or x.shape[2] != 3:
        raise ValueError(f" (H,W,3)， {x.shape}: {path.name}")
    return np.clip(x.astype(np.float32, copy=False), 0.0, 1.0)


def show_grid(rows: list[tuple[str, list[Path]]], ncols: int) -> None:
    nrows = len(rows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.4 * nrows))
    if nrows == 1:
        axes = np.expand_dims(axes, axis=0)
    for r, (title, plist) in enumerate(rows):
        for c in range(ncols):
            ax = axes[r, c]
            if c < len(plist):
                img = load_rgb_npy(plist[c])
                ax.imshow(img, vmin=0.0, vmax=1.0)
            ax.axis("off")
        axes[r, 0].text(
            -0.35,
            0.5,
            title,
            transform=axes[r, 0].transAxes,
            fontsize=11,
            fontweight="bold",
            va="center",
            ha="right",
        )
    plt.tight_layout()
    plt.show()


show_grid(
    [
        ("globalwheat", sel_gw),
        ("rgb_bonirob", sel_rb),
    ],
    N_PER_CLASS,
)

In [ ]:
from PIL import Image


def find_ews_train_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        d = p / "EWS" / "data" / "EWS-Dataset" / "train"
        if d.is_dir():
            return d
    raise FileNotFoundError(" EWS/data/EWS-Dataset/train")


EWS_TRAIN_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\EWS-Dataset\train")
if not EWS_TRAIN_DIR.is_dir():
    EWS_TRAIN_DIR = find_ews_train_dir()
print("EWS_TRAIN_DIR:", EWS_TRAIN_DIR)


def list_ews_rgb_pngs(train_dir: Path) -> list[Path]:
    out: list[Path] = []
    for p in sorted(train_dir.glob("*.png")):
        if p.name.endswith("_mask.png"):
            continue
        if not p.with_name(p.stem + "_mask.png").is_file():
            raise FileNotFoundError(f": {p.name}")
        out.append(p)
    if not out:
        raise FileNotFoundError(f" RGB PNG: {train_dir}")
    return out


def rgb_u8_to_f32(rgb: np.ndarray) -> np.ndarray:
    return rgb.astype(np.float32) / 255.0


def mean_brightness_saturation(img_f: np.ndarray) -> tuple[float, float]:
    """img_f: (H, W, 3), float32, [0, 1]。"""
    r = img_f[..., 0]
    g = img_f[..., 1]
    b = img_f[..., 2]
    y = 0.299 * r + 0.587 * g + 0.114 * b
    mx = np.maximum(np.maximum(r, g), b)
    mn = np.minimum(np.minimum(r, g), b)
    sat = np.zeros_like(mx, dtype=np.float32)
    ok = mx > 1e-8
    sat[ok] = (mx[ok] - mn[ok]) / mx[ok]
    return float(np.mean(y)), float(np.mean(sat))


N_EWS = 10
SEED_EWS = 43
rng_ews = random.Random(SEED_EWS)
ews_paths = list_ews_rgb_pngs(EWS_TRAIN_DIR)
sel_ews = pick_paths(ews_paths, N_EWS, rng_ews)

rows: list[tuple[str, int, int, float, float]] = []
fig, axes = plt.subplots(2, 5, figsize=(14, 5.5))
for ax, pth in zip(np.ravel(axes), sel_ews):
    im = Image.open(pth).convert("RGB")
    arr_u8 = np.asarray(im)
    br, sat = mean_brightness_saturation(rgb_u8_to_f32(arr_u8))
    rows.append((pth.name, im.width, im.height, br, sat))
    ax.imshow(arr_u8)
    ax.axis("off")

plt.tight_layout()
plt.show()

print(f"{'':<52} {'':>5} {'':>5} {'Y':>10} {'S':>10}")
print("-" * 88)
for name, w, h, br, sat in rows:
    print(f"{name:<52} {w:5d} {h:5d} {br:10.4f} {sat:10.4f}")

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

CROP350_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\sugar-beets-2016-DatasetNinja\ds\img_350_crop")


def _rgb_u8_to_f32(rgb: np.ndarray) -> np.ndarray:
    return rgb.astype(np.float32) / 255.0


def _mean_brightness_saturation(img_f: np.ndarray) -> tuple[float, float]:
    r = img_f[..., 0]
    g = img_f[..., 1]
    b = img_f[..., 2]
    y = 0.299 * r + 0.587 * g + 0.114 * b
    mx = np.maximum(np.maximum(r, g), b)
    mn = np.minimum(np.minimum(r, g), b)
    sat = np.zeros_like(mx, dtype=np.float32)
    ok = mx > 1e-8
    sat[ok] = (mx[ok] - mn[ok]) / mx[ok]
    return float(np.mean(y)), float(np.mean(sat))


if not CROP350_DIR.is_dir():
    raise FileNotFoundError(f": {CROP350_DIR}")

crop_paths = sorted(CROP350_DIR.glob("rgb_*.png"))
if not crop_paths:
    raise FileNotFoundError(f" rgb_*.png: {CROP350_DIR}")

rows_bs: list[tuple[str, float, float]] = []
from tqdm import tqdm as _tqdm
_iter = _tqdm(crop_paths, desc="img_350_crop") if _tqdm else crop_paths
for pth in _iter:
    im = Image.open(pth).convert("RGB")
    if im.size != (350, 350):
        raise ValueError(f"350×350， {im.size[0]}×{im.size[1]}: {pth.name}")
    br, sat = _mean_brightness_saturation(_rgb_u8_to_f32(np.asarray(im)))
    rows_bs.append((pth.name, br, sat))

br_all = np.array([r[1] for r in rows_bs], dtype=np.float64)
sat_all = np.array([r[2] for r in rows_bs], dtype=np.float64)

print(f": {CROP350_DIR}")
print(f": {len(rows_bs)}")
print(
 f" Y —  {br_all.mean():.4f}   {br_all.std():.4f}   {br_all.min():.4f}   {br_all.max():.4f}"
)
print(
    f" S —  {sat_all.mean():.4f}   {sat_all.std():.4f}   {sat_all.min():.4f}   {sat_all.max():.4f}"
)
print("\n 10 （）:")
print(f"{'':<42} {'Y':>10} {'S':>10}")
print("-" * 64)
for name, br, sat in rows_bs[:10]:
    print(f"{name:<42} {br:10.4f} {sat:10.4f}")

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

EWS_TRAIN_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\EWS-Dataset\train")
CROP350_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\sugar-beets-2016-DatasetNinja\ds\img_350_crop")
N_SAMPLES = 10
SEED_MATCH = 44


def _rgb_u8_to_f32(a: np.ndarray) -> np.ndarray:
    return a.astype(np.float32) / 255.0


def _mean_luma_bt601(rgb_f: np.ndarray) -> float:
    r = rgb_f[..., 0]
    g = rgb_f[..., 1]
    b = rgb_f[..., 2]
    y = 0.299 * r + 0.587 * g + 0.114 * b
    return float(np.mean(y))


def _match_mean_luma(rgb_f: np.ndarray, y_target: float) -> np.ndarray:
    y_m = _mean_luma_bt601(rgb_f)
    s = y_target / (y_m + 1e-8)
    return np.clip(rgb_f * s, 0.0, 1.0)


def _list_ews_rgb(train_dir: Path) -> list[Path]:
    out: list[Path] = []
    for p in sorted(train_dir.glob("*.png")):
        if p.name.endswith("_mask.png"):
            continue
        if not p.with_name(p.stem + "_mask.png").is_file():
            continue
        out.append(p)
    return out


ews_rgb = _list_ews_rgb(EWS_TRAIN_DIR)
if not ews_rgb:
    raise FileNotFoundError(f" EWS RGB: {EWS_TRAIN_DIR}")

y_ews_per_image = []
for p in ews_rgb:
    im = Image.open(p).convert("RGB")
    y_ews_per_image.append(_mean_luma_bt601(_rgb_u8_to_f32(np.asarray(im))))
Y_TARGET = float(np.mean(y_ews_per_image))
print(f"EWS train  {len(ews_rgb)} ， → Y_target = {Y_TARGET:.4f}")

crop_all = sorted(CROP350_DIR.glob("rgb_*.png"))
if len(crop_all) < N_SAMPLES:
    raise FileNotFoundError(f"{CROP350_DIR}  rgb_*.png  {N_SAMPLES} ")

rng_m = random.Random(SEED_MATCH)
sel_crop = rng_m.sample(crop_all, N_SAMPLES)

fig, axes = plt.subplots(2, N_SAMPLES, figsize=(2.2 * N_SAMPLES, 4.8))
for j, pth in enumerate(sel_crop):
    im = Image.open(pth).convert("RGB")
    if im.size != (350, 350):
        raise ValueError(im.size)
    rgb0 = _rgb_u8_to_f32(np.asarray(im))
    y0 = _mean_luma_bt601(rgb0)
    rgb1 = _match_mean_luma(rgb0, Y_TARGET)
    y1 = _mean_luma_bt601(rgb1)

    ax0, ax1 = axes[0, j], axes[1, j]
    ax0.imshow(rgb0, vmin=0, vmax=1)
    ax0.axis("off")
    ax1.imshow(rgb1, vmin=0, vmax=1)
    ax1.axis("off")

axes[0, 0].text(
    -0.25,
    0.5,
    "img_350_crop\n",
    transform=axes[0, 0].transAxes,
    fontsize=10,
    fontweight="bold",
    va="center",
    ha="right",
)
axes[1, 0].text(
    -0.25,
    0.5,
    "\n(EWS )",
    transform=axes[1, 0].transAxes,
    fontsize=10,
    fontweight="bold",
    va="center",
    ha="right",
)

    f" 350×350： EWS train （Y_target={Y_TARGET:.4f}）",
    fontsize=11,
)
plt.tight_layout()
plt.show()

print(f"{'':<40} {'Y':>8} {'Y ':>10} {' s':>8}")
print("-" * 70)
for pth in sel_crop:
    im = Image.open(pth).convert("RGB")
    rgb0 = _rgb_u8_to_f32(np.asarray(im))
    y0 = _mean_luma_bt601(rgb0)
    rgb1 = _match_mean_luma(rgb0, Y_TARGET)
    y1 = _mean_luma_bt601(rgb1)
    s = Y_TARGET / (y0 + 1e-8)
    print(f"{pth.name:<40} {y0:8.4f} {y1:10.4f} {s:8.3f}")

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image

EWS_TRAIN_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\EWS-Dataset\train")
CROP350_DIR = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\sugar-beets-2016-DatasetNinja\ds\img_350_crop")
OUT_CROP350_EWS_Y = CROP350_DIR.parent / "img_350_crop_ews_y"


def _rgb_u8_to_f32(a: np.ndarray) -> np.ndarray:
    return a.astype(np.float32) / 255.0


def _mean_luma_bt601(rgb_f: np.ndarray) -> float:
    r = rgb_f[..., 0]
    g = rgb_f[..., 1]
    b = rgb_f[..., 2]
    y = 0.299 * r + 0.587 * g + 0.114 * b
    return float(np.mean(y))


def _match_mean_luma(rgb_f: np.ndarray, y_target: float) -> np.ndarray:
    y_m = _mean_luma_bt601(rgb_f)
    s = y_target / (y_m + 1e-8)
    return np.clip(rgb_f * s, 0.0, 1.0)


def _list_ews_rgb(train_dir: Path) -> list[Path]:
    out: list[Path] = []
    for p in sorted(train_dir.glob("*.png")):
        if p.name.endswith("_mask.png"):
            continue
        if not p.with_name(p.stem + "_mask.png").is_file():
            continue
        out.append(p)
    return out


OUT_CROP350_EWS_Y.mkdir(parents=True, exist_ok=True)

ews_rgb = _list_ews_rgb(EWS_TRAIN_DIR)
if not ews_rgb:
    raise FileNotFoundError(f" EWS RGB: {EWS_TRAIN_DIR}")

y_ews_per_image = []
for p in ews_rgb:
    im = Image.open(p).convert("RGB")
    y_ews_per_image.append(_mean_luma_bt601(_rgb_u8_to_f32(np.asarray(im))))
Y_TARGET = float(np.mean(y_ews_per_image))
print(f"Y_target（EWS train ）= {Y_TARGET:.4f}")

crop_paths = sorted(CROP350_DIR.glob("rgb_*.png"))
if not crop_paths:
    raise FileNotFoundError(f" rgb_*.png: {CROP350_DIR}")

from tqdm import tqdm as _tqdm_b
_iter_b = _tqdm_b(crop_paths, desc="img_350_crop → ews_y") if _tqdm_b else crop_paths

n_ok = 0
for pth in _iter_b:
    im = Image.open(pth).convert("RGB")
    if im.size != (350, 350):
        raise ValueError(f" 350×350: {pth.name} → {im.size}")
    rgb0 = _rgb_u8_to_f32(np.asarray(im))
    rgb1 = _match_mean_luma(rgb0, Y_TARGET)
    out_u8 = np.clip(np.round(rgb1 * 255.0), 0, 255).astype(np.uint8)
    out_path = OUT_CROP350_EWS_Y / pth.name
    Image.fromarray(out_u8, mode="RGB").save(out_path, format="PNG", optimize=True)
    n_ok += 1

print(f": {n_ok}  → {OUT_CROP350_EWS_Y}")